In [ ]:
from transformers import AutoModelForTokenClassification,AutoTokenizer
from peft import PeftModel
import torch.nn.functional as F
import numpy as np
import re
import pickle
import pandas as pd
import warnings
import joblib
import torch

warnings.filterwarnings("ignore")

In [ ]:
def translate(seq):
    """Translate to seq to amino acid"""
    table = {
        'ATA': 'I', 'ATC': 'I', 'ATT': 'I', 'ATG': 'M',
        'ACA': 'T', 'ACC': 'T', 'ACG': 'T', 'ACT': 'T',
        'AAC': 'N', 'AAT': 'N', 'AAA': 'K', 'AAG': 'K',
        'AGC': 'S', 'AGT': 'S', 'AGA': 'R', 'AGG': 'R',
        'CTA': 'L', 'CTC': 'L', 'CTG': 'L', 'CTT': 'L',
        'CCA': 'P', 'CCC': 'P', 'CCG': 'P', 'CCT': 'P',
        'CAC': 'H', 'CAT': 'H', 'CAA': 'Q', 'CAG': 'Q',
        'CGA': 'R', 'CGC': 'R', 'CGG': 'R', 'CGT': 'R',
        'GTA': 'V', 'GTC': 'V', 'GTG': 'V', 'GTT': 'V',
        'GCA': 'A', 'GCC': 'A', 'GCG': 'A', 'GCT': 'A',
        'GAC': 'D', 'GAT': 'D', 'GAA': 'E', 'GAG': 'E',
        'GGA': 'G', 'GGC': 'G', 'GGG': 'G', 'GGT': 'G',
        'TCA': 'S', 'TCC': 'S', 'TCG': 'S', 'TCT': 'S',
        'TTC': 'F', 'TTT': 'F', 'TTA': 'L', 'TTG': 'L',
        'TAC': 'Y', 'TAT': 'Y', 'TAA': '<eos>', 'TAG': '<eos>',
        'TGC': 'C', 'TGT': 'C', 'TGA': '<eos>', 'TGG': 'W',
    }
    protein = ""

    for i in range(0, len(seq), 3):

        codon = seq[i:i + 3]
        if len(codon)<3:
           return protein

        protein += table[codon]
    
    return protein

In [ ]:
def metaPred(model,tokenizer,modelesm,tokesm,seq,codons,metamodel):
    '''Predict TIS with all three models
       codons should be a list of any of the ten possible start codons 
    '''
    allprobs=[]
    for i in range(3):
        s=seq[i:len(seq)]
        aa=translate(s)

        for c in codons:
            indices=[m.start() for m in re.finditer(c, s)]

            ### NT
            inputs=tokenizer(s,truncation=True)
            tokens = tokenizer.convert_ids_to_tokens(
            inputs['input_ids']
            )

            attention_mask = torch.tensor([inputs["attention_mask"]])
            input_ids = torch.tensor([inputs["input_ids"]])

            with torch.no_grad():

                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                ).logits
                probs=F.sigmoid(logits).squeeze().tolist()

            ### ESM2
            inputsesm=tokesm(aa)

            attention_maskesm = torch.tensor([inputsesm["attention_mask"]])
            input_idsesm = torch.tensor([inputsesm["input_ids"]])

            with torch.no_grad():

                logitsesm = modelesm(
                    input_ids=input_idsesm,
                    attention_mask=attention_maskesm
                ).logits
                probsesm=F.sigmoid(logitsesm).squeeze().tolist()
            
            for j in indices:
                if j%3==0:
                    idx=j+i
                    ti=int(j//3)+1
                    allprobs.append([tokens[ti],i+1,idx,probs[ti],probsesm[ti]])

    allprobs=pd.DataFrame(allprobs)
    allprobs.columns=['Codon','Reading Frame','Start Site','NT','ESM2']

    ##### For missing columns
    allcols=['NTPreds',
    'ESMPreds',
    'Codon_AAG',
    'Codon_ACG',
    'Codon_AGG',
    'Codon_ATA',
    'Codon_ATC',
    'Codon_ATG',
    'Codon_ATT',
    'Codon_CTG',
    'Codon_GTG',
    'Codon_NotStartCodon',
    'Codon_TTG']
    keepcods=['ATG', 'CTG', 'GTG', 'AAG', 'AGG', 'TTG', 'ACG', 'ATC', 'ATA','ATT']
    ncs=[]
    for i in allprobs['Codon']:
        if i in keepcods:
            ncs.append(i)
        else:
            ncs.append('NotStartCodon')
    allprobs['Codon']=ncs

    X=allprobs[['NT','ESM2','Codon']]
    X.columns=['NTPreds','ESMPreds','Codon'] ## meta model columns

    
    X=pd.get_dummies(X)
    miss=set(allcols)-set(X.columns)
    for i in miss:
        X[i]=0
    X=X[allcols]

    mpreds=list(metamodel.predict_proba(X)[:,1])
    allprobs['MetaTIS']=mpreds
    allprobs=allprobs.sort_values(by='MetaTIS',ascending=False).reset_index(drop=True)

    if allprobs.shape[0]==0:
        return None
    else:
        return allprobs

In [ ]:
### load NT model
base_model_name = "InstaDeepAI/nucleotide-transformer-v2-50m-3mer-multi-species"
checkpoint_dir = "../Models/NToptimalCheckpoint" 

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)

# Load base model
base_model = AutoModelForTokenClassification.from_pretrained(
    base_model_name,
    trust_remote_code=True,
    num_labels=1
)

# Load LoRA adapters
model = PeftModel.from_pretrained(base_model, checkpoint_dir)

# Put model in eval mode
model.eval()

In [ ]:
############# load ESM2 model
# Paths
base_model_name = "facebook/esm2_t12_35M_UR50D"
checkpoint_dir = "../Models/ESM2optimalCheckpoint"

# Load tokenizer
tokenizeresm = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)

# Load base model
base_modelesm = AutoModelForTokenClassification.from_pretrained(
    base_model_name,
    trust_remote_code=True,
    num_labels=1
)

# Load LoRA adapters
modelesm = PeftModel.from_pretrained(base_modelesm, checkpoint_dir)

# Put model in eval mode
modelesm.eval()

In [ ]:
## Load metamodel
model_path = '../Models/calibrated_meta_model.joblib'
metamodel = joblib.load(model_path)

In [ ]:
test_sequence="GAGCCAACATGGCAGCGGGGTTCGGGCGATGCTGCAGGTGTTCTTTACAGGTCCTGAGAAGTATTTCTCGTTTTCATTGGAGATCACAGCATACAAAAGCCAATCGACAACGTGAACCAGGATTAGGATTTAGTTTTGAGTTCACCGAACAGCAGAAAGAATTTCAAGCTACTGCTCGTAAATTTGCCAGAGAGGAAATCATCCCAGTGGCTGCAGAATATGATAAAACTGGTGAATATCCAGTCCCCCTAATTAGAAGAGCCTGGGAACTTGGTTTAATGAACACACACATTCCAGAGAACTGTGGAGGTCTTGGACTTGGAACTTTTGATGCTTGTTTAATTAGTGAAGAATTGGCTTATGGATGTACAGGGGTTCAGACTGCTATTGAAGGAAATTCTTTGGGGCAAATGCCTATTATTATTGCTGGAAATGATCAACAAAAGAAGAAGTATTTGGGGAGAATGACTGAGGAGCCATTGATGTGTGCTTATTGTGTAACAGAACCTGGAGCAGGCTCTGATGTAGCTGGTATAAAGACCAAAGCAGAAAAGAAAGGAGATGAGTATATTATTAATGGTCAGAAGATGTGGATAACCAACGGAGGAAAAGCTAATTGGTATTTTTTATTGGCACGTTCTGATCCAGATCCTAAAGCTCCTGCTAATAAAGCCTTTACTGGATTCATTGTGGAAGCAGATACCCCAGGAATTCAGATTGGGAGAAAGGAATTAAACATGGGCCAGCGATGTTCAGATACTAGAGGAATTGTCTTCGAAGATGTGAAAGTGCCTAAAGAAAATGTTTTAATTGGTGACGGAGCTGGTTTCAAAGTTGCAATGGGAGCTTTTGATAAAACCAGACCTGTAGTAGCTGCTGGTGCTGTTGGATTAGCACAAAGAGCTTTGGATGAAGCTACCAAGTATGCCCTGGAAAGGAAAACTTTCGGAAAGCTACTTGTAGAGCACCAAGCAATATCATTTATGCTGGCTGAAATGGCAATGAAAGTTGAACTAGCTAGAATGAGTTACCAGAGAGCAGCTTGGGAGGTTGATTCTGGTCGTCGAAATACCTATTATGCTTCTATTGCAAAGGCATTTGCTGGAGATATTGCAAATCAGTTAGCTACTGATGCTGTGCAGATACTTGGAGGCAATGGATTTAATACAGAATATCCTGTAGAAAAACTAATGAGGGATGCCAAAATCTATCAGATTTATGAAGGTACTTCACAAATTCAAAGACTTATTGTAGCCCGTGAACACATTGACAAGTACAAAAATTAA"

results=metaPred(model,tokenizer,modelesm,tokenizeresm,test_sequence,["ATG","CTG"],metamodel)

results.head()